## We will construct a linear model that can predict a car's mileage (mpg) by using its other attributes.

### Data Description: 

The dataset has 6 variables, including the name of the car and its various attributes like horsepower, weight etc. Missing values in the data are marked by a series of question marks.

A detailed description of the variables is given below.

1. mpg: miles per gallon
2. cylinders: number of cylinders
3. displacement: engine displacement in cubic inches
4. horsepower: horsepower of the car
5. weight: weight of the car in pounds
6. acceleration: time taken, in seconds, to accelerate from O to 60 mph


### Importing Libraries

In [1]:
import pandas as pd
import numpy as np

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
from sklearn.linear_model import LinearRegression

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

C:\Users\codyd\AppData\Local\Temp\ipykernel_21480\4223966418.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### Load the data

In [2]:
# df = pd.read_csv('/content/AUTOMPG.csv')
df = pd.read_csv('AUTOMPG.csv')

### Displaying the first few rows of the dataset

In [3]:
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration
0,18.0,8,307.0,130.0,3504,12.0
1,15.0,8,350.0,165.0,3693,11.5
2,18.0,8,318.0,150.0,3436,11.0
3,16.0,8,304.0,150.0,3433,12.0
4,17.0,8,302.0,140.0,3449,10.5


### Checking the shape of the dataset

In [4]:
df.shape

(398, 6)

* There are 398 rows and 6 columns in the data

### Data Preparation for modeling

In [5]:
# independent variables
X = df.drop(["mpg"], axis=1)
# dependent variable
y = df["mpg"]

**We will now split X and y into train and test sets in a 70:30 ratio.**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=1
)

## Model Building - Linear Regression

In [7]:
lr=LinearRegression()
lr.fit(X_train,y_train)

LinearRegression()

### Model Performance Check

**Let's check the performance of the model using different metrics.**

* We will be using metric functions defined in sklearn for RMSE, MAE, and $R^2$.
* We will define a function to calculate MAPE and adjusted $R^2$.    
* We will create a function which will print out all the above metrics in one go.

In [8]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE
    
    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

def saveTestResults(model, predictors, target, data, modelName):

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE

    # Storing results 
    data['Model'].append(modelName)
    data['RMSE'].append(rmse)
    data['MAE'].append(mae)
    data['R-squared'].append(r2)
    data['Adj. R-squared'].append(adjr2)
    data['MAPE'].append(mape)

    return data

In [9]:
# Instantiating dict to hold results
resultsDict = {
    'Model': [],
    'RMSE': [],
    'MAE': [],
    'R-squared': [],
    'Adj. R-squared': [],
    'MAPE': [],
    'Overfit': []
}

In [10]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
model_train_perf = model_performance_regression(lr, X_train, y_train)
model_train_perf

Training Performance



,RMSE,MAE,R-squared,Adj. R-squared,MAPE
0,4.402643,3.344707,0.687252,0.681503,14.635446


In [11]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
saveTestResults(lr, X_test, y_test, resultsDict, 'LinearRegression')
model_test_perf = model_performance_regression(lr, X_test, y_test)
model_test_perf

Test Performance



,RMSE,MAE,R-squared,Adj. R-squared,MAPE
0,3.834676,2.988926,0.748495,0.737464,12.914085


In [12]:
overfit = model_train_perf['MAPE'][0] - model_test_perf['MAPE'][0]

if overfit < 0:
    overfit = abs(overfit)

resultsDict['Overfit'].append(overfit)

**Observations**

- MAE indicates that our current model is able to predict MPG within a mean error of ~2.988  on test data.
- The RMSE values are higher than the MAE values as the squares of residuals penalizes the model more for larger errors in prediction.
- MAPE of ~12.91 on the test data indicates that the model can predict within ~12.91% of the MPG.

## Support Vector Regression - SVR

In [13]:
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [14]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=1)

# Initialize SVR model
svr_model = SVR(kernel='linear')  # Specify the kernel type, e.g., 'linear', 'rbf', 'poly', etc.

# Train the SVR model
svr_model.fit(X_train, y_train)

SVR(kernel='linear')

In [15]:
# Make predictions on the testing set
y_pred = svr_model.predict(X_train)

In [16]:
errors = abs(y_pred - y_train)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_train, y_pred)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_train, y_pred)

# R-squared (R²)
r_squared = r2_score(y_train, y_pred)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

train_mape = 100 * (errors / y_train)
accuracy = 100 - np.mean(train_mape)
print('Accuracy:', round(accuracy, 2), '%.')

Mean Squared Error: 29.62228818936051
Root Mean Squared Error: 5.442636143392328
Mean Absolute Error: 4.344718325233325
R-squared: 0.5220464555150399
Accuracy: 79.17 %.


In [17]:
model_train_perf = model_performance_regression(svr_model, X_train, y_train)
print(model_train_perf)

       RMSE       MAE  R-squared  Adj. R-squared       MAPE
0  5.442636  4.344718   0.522046        0.513261  20.826641


In [18]:

# Make predictions on the testing set
y_pred = svr_model.predict(X_test)

In [19]:
errors = abs(y_pred - y_test)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, y_pred)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, y_pred)

# R-squared (R²)
r_squared = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

test_mape = 100 * (errors / y_test)
accuracy = 100 - np.mean(test_mape)
overfit = train_mape - test_mape
print('Accuracy:', round(accuracy, 2), '%.')
print('Overfit:', overfit)

Mean Squared Error: 23.41973151348753
Root Mean Squared Error: 4.839393713419846
Mean Absolute Error: 3.6989338617090937
R-squared: 0.5994363507670926
Accuracy: 82.45 %.
Overfit: 0     NaN
1     NaN
2     NaN
3     NaN
4     NaN
       ..
393   NaN
394   NaN
395   NaN
396   NaN
397   NaN
Name: mpg, Length: 398, dtype: float64


In [20]:
saveTestResults(svr_model, X_test, y_test, resultsDict, "SupportVectorRegression")
model_test_perf = model_performance_regression(svr_model, X_test, y_test)
print(model_test_perf)

       RMSE       MAE  R-squared  Adj. R-squared       MAPE
0  4.839394  3.698934   0.599436        0.581868  17.552366


In [21]:
overfit = model_train_perf['MAPE'][0] - model_test_perf['MAPE'][0]

if overfit < 0:
    overfit = abs(overfit)

resultsDict['Overfit'].append(overfit)

<b>Observations:</b>
<ul>
    <li>Test performed better than training</li>
    <li>Able to predict within a mean error of ~3.7, considerably worse than Linear Regression</li>
    <li>MAPE of 20.8 on training, 17.5 on test, also much worse than Linear Regression</li>
    <li>Test results show stronger numbers in all categories than test, but worse in all categories, compared to Linear Regression, by a significant margin</li>
</ul>

### Random Forest

In [22]:
from sklearn.ensemble import RandomForestRegressor

In [23]:
rf = RandomForestRegressor(n_estimators=1000, random_state=35)
rf.fit(X_train, y_train)

RandomForestRegressor(n_estimators=1000, random_state=35)

In [24]:
preds = rf.predict(X_train)
errors = abs(preds - y_train)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_train, preds)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_train, preds)

# R-squared (R²)
r_squared = r2_score(y_train, preds)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

mape = 100 * (errors / y_train)
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

Mean Squared Error: 2.3251388448230474
Root Mean Squared Error: 1.524840596529043
Mean Absolute Error: 1.0642719424460316
R-squared: 0.9624840476468663
Accuracy: 95.5 %.


In [25]:
model_train_perf = model_performance_regression(rf, X_train, y_train)
print(model_train_perf)

       RMSE       MAE  R-squared  Adj. R-squared      MAPE
0  1.524841  1.064272   0.962484        0.961794  4.504629


In [26]:
preds = rf.predict(X_test)
errors = abs(preds - y_test)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, preds)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, preds)

# R-squared (R²)
r_squared = r2_score(y_test, preds)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

mape = 100 * (errors / y_test)
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

Mean Squared Error: 11.337965253499954
Root Mean Squared Error: 3.367189518500548
Mean Absolute Error: 2.4349691666666566
R-squared: 0.8060790434680118
Accuracy: 89.85 %.


In [27]:
saveTestResults(rf, X_test, y_test, resultsDict, "RandomForest")
model_test_perf = model_performance_regression(rf, X_test, y_test)
print(model_test_perf)

      RMSE       MAE  R-squared  Adj. R-squared       MAPE
0  3.36719  2.434969   0.806079        0.797574  10.146406


In [28]:
overfit = model_train_perf['MAPE'][0] - model_test_perf['MAPE'][0]

if overfit < 0:
    overfit = abs(overfit)

resultsDict['Overfit'].append(overfit)

<b>Observations:</b>
<ul>
    <li>Test results significantly worse than training results; indication of overfit to training data</li>
    <li>Aside from obvious overfit, the model's numbers are the strongest overall, thus far</li>
    <li>Unsure of how to interpret these results, however, due to overfitness</li>
</ul>

### Decision Tree

In [29]:
from sklearn.tree import DecisionTreeRegressor

In [30]:
dt = DecisionTreeRegressor(random_state=35)
dt.fit(X_train, y_train)

DecisionTreeRegressor(random_state=35)

In [31]:
preds = dt.predict(X_train)
errors = abs(preds - y_train)

In [32]:
# Mean Squared Error (MSE)
mse = mean_squared_error(y_train, preds)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_train, preds)

# R-squared (R²)
r_squared = r2_score(y_train, preds)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

mape = 100 * (errors / y_train)
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

Mean Squared Error: 0.01618705035971223
Root Mean Squared Error: 0.12722833945199563
Mean Absolute Error: 0.01079136690647482
R-squared: 0.9997388230765725
Accuracy: 99.93 %.


In [33]:
model_train_perf = model_performance_regression(dt, X_train, y_train)

In [34]:
preds = dt.predict(X_test)
errors = abs(preds - y_test)

# Mean Squared Error (MSE)
mse = mean_squared_error(y_test, preds)

# Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)

# Mean Absolute Error (MAE)
mae = mean_absolute_error(y_test, preds)

# R-squared (R²)
r_squared = r2_score(y_test, preds)

print("Mean Squared Error:", mse)
print("Root Mean Squared Error:", rmse)
print("Mean Absolute Error:", mae)
print("R-squared:", r_squared)

mape = 100 * (errors / y_test)
accuracy = 100 - np.mean(mape)
print('Accuracy:', round(accuracy, 2), '%.')

Mean Squared Error: 20.534499999999998
Root Mean Squared Error: 4.531500855125153
Mean Absolute Error: 3.276666666666667
R-squared: 0.648784434166715
Accuracy: 86.29 %.


In [35]:
saveTestResults(dt, X_test, y_test, resultsDict, "DecisionTree")
model_test_perf = model_performance_regression(dt, X_test, y_test)
print(model_test_perf)

       RMSE       MAE  R-squared  Adj. R-squared       MAPE
0  4.531501  3.276667   0.648784         0.63338  13.714864


In [36]:
overfit = model_train_perf['MAPE'][0] - model_test_perf['MAPE'][0]

if overfit < 0:
    overfit = abs(overfit)

resultsDict['Overfit'].append(overfit)

<b>Observations:</b>
<ul>
    <li>Monumental overfit to training data</li>
    <li>Unsurprising underforpermance in test, due to overfit</li>
    <li>Results of test most likely irrelevant, due to extreme overfit</li>
</ul>

## Conclusion

In [37]:
results_df = pd.DataFrame(resultsDict)

In [41]:
styled_df = results_df.style.background_gradient(cmap='coolwarm')
styled_df

,Model,RMSE,MAE,R-squared,Adj. R-squared,MAPE,Overfit
0,LinearRegression,3.834676,2.988926,0.748495,0.737464,12.914085,1.721361
1,SupportVectorRegression,4.839394,3.698934,0.599436,0.581868,17.552366,3.274275
2,RandomForest,3.367190,2.434969,0.806079,0.797574,10.146406,5.641777
3,DecisionTree,4.531501,3.276667,0.648784,0.633380,13.714864,13.644584


<b>Observations:</b>
<ul>
    <li>Decision Tree is the most overfit, by far, followed by Random Forest, then SVR</li>
    <li>Linear Regression has the lowest indicated overfit, by far</li>
    <li>Random forest has the lowest RMSE, MAE, and MAPE, but some indication of overfit</li>
    <li>SVR has the highest RMSE, MAE, and MAPE, as well as the lowest R-squared and Adj. R-squared</li>
</ul>

#### Final Word

Overall, it is hard to determine which model is objectively the best, based on the data above. Just glancing at the numbers, Random Forest would appear to be the strongest, as it is the strongest in all categories, aside from overfitness. This could partially be due to the relatively small size of the dataset; perhaps its overfitness value would be lower, if the dataset were larger. If this were the case, Random Forest would obviously be the best. However, Linear Regression is comparable to Random Forest in every metric, but also has the lowest overfit value by a large margin, which leads me to believe that it is the strongest for this particular dataset.